In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import sys
import os

sys.path.append('..')
os.chdir('..')  

from src.utils.config_loader import load_config
from src.data_pipeline.preprocess import *
from src.data_pipeline.features import *
config = load_config("configs/data_config.yaml")
print("Config loaded successfully ✅")

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
Config loaded successfully ✅


In [2]:
# Load ratings data
raw_path = config['paths']['raw_data']

df_ratings = pd.read_parquet(raw_path + "Electronics_ratings.parquet")

print(f"Shape: {df_ratings.shape}")
print(f"\nColumns: {df_ratings.columns.tolist()}")
print(f"\nFirst 5 rows:")
df_ratings.head()

Shape: (1000000, 10)

Columns: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

First 5 rows:


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,"[{'attachment_type': 'IMAGE', 'large_image_url...",B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True


In [3]:
prep_config = config['preprocessing']

df_ratings = drop_useless_columns(df_ratings, prep_config['drop_columns'])
df_ratings = remove_missing_values(df_ratings, prep_config['missing_values']['subset'])
df_ratings = remove_duplicates(df_ratings)
df_ratings = convert_timestamp(df_ratings)
df_ratings = convert_to_integer(df_ratings, prep_config['convert_to_integer']['columns'])
df_ratings = handle_outliers(df_ratings, prep_config['outliers']['columns'])
df_ratings = detect_spam(df_ratings, prep_config['spam_detection']['max_reviews_per_day'], prep_config['spam_detection']['min_time_gap'])
df_ratings = add_review_weight(df_ratings)
df_ratings = filter_text(df_ratings,column='text',
                         min_words = prep_config['text_filter']['min_words'],
                         max_words = prep_config['text_filter']['max_words'])
df_ratings, encoders = encode_labels(df_ratings, prep_config['encode']['columns'])
train_df, val_df, test_df = time_based_split(
    df_ratings, 
    column="timestamp", 
    val_year=prep_config['split']['val_year'], 
    test_year=prep_config['split']['test_year']
)

[INFO] Dropped columns: ['images']
[INFO] Removed 220 rows with missing values
[INFO] Removed 2495 duplicate rows
[INFO] Converted 'timestamp' to datetime
[INFO] Converted 'verified_purchase' to integer
[INFO] Applied Log Transformation (log1p) to 'helpful_vote'
[INFO] Removed 7690 spam users
[INFO] Added weight column: verified=1.0, unverified=0.7
[INFO] Valid texts for NLP: 719978
[INFO] Marked 79233 short texts as None
[INFO] Truncated 26563 long texts to 250 words
[INFO] Encoded 'user_id' — 177495 unique labels
[INFO] Encoded 'asin' — 274155 unique labels
[INFO] Train: 608593 rows (76.1%)
[INFO] Val:   92463 rows (11.6%)
[INFO] Test:  98155 rows (12.3%)


In [4]:
train_df = add_user_segment(train_df)
train_df = add_features(train_df)
train_df, scaler = normalize(train_df, 'helpful_vote')

val_df = add_user_segment(val_df)
val_df = add_features(val_df)
val_df, _ = normalize(val_df, 'helpful_vote', scaler=scaler)

test_df = add_user_segment(test_df)
test_df = add_features(test_df)
test_df, _ = normalize(test_df, 'helpful_vote', scaler=scaler)

[INFO] User segments:
user_segment
Medium    280082
Heavy     214735
Light     113776
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Fitted and normalized 'helpful_vote'
[INFO] User segments:
user_segment
Light     52463
Medium    33929
Heavy      6071
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Transformed 'helpful_vote' using existing scaler
[INFO] User segments:
user_segment
Light     48649
Medium    37819
Heavy     11687
Name: count, dtype: int64
[INFO] Added features: user_verified_ratio, item_avg_rating, is_weekend
[INFO] Transformed 'helpful_vote' using existing scaler


In [5]:
total_users = len(encoders['user_id'].classes_)
total_items = len(encoders['asin'].classes_)

print(f"[INFO] Total Unique Users: {total_users}")
print(f"[INFO] Total Unique Items: {total_items}\n")

print("=== Train Matrix ===")
train_matrix = build_user_item_matrix(
    df=train_df, 
    total_users=total_users, 
    total_items=total_items
)

print("\n=== Validation Matrix ===")
val_matrix = build_user_item_matrix(
    df=val_df, 
    total_users=total_users, 
    total_items=total_items
)

print("\n=== Test Matrix ===")
test_matrix = build_user_item_matrix(
    df=test_df, 
    total_users=total_users, 
    total_items=total_items
)

[INFO] Total Unique Users: 177495
[INFO] Total Unique Items: 274155

=== Train Matrix ===
[INFO] Matrix built successfully with shape: (177495, 274155)
[INFO] Matrix sparsity: 99.9987%

=== Validation Matrix ===
[INFO] Matrix built successfully with shape: (177495, 274155)
[INFO] Matrix sparsity: 99.9998%

=== Test Matrix ===
[INFO] Matrix built successfully with shape: (177495, 274155)
[INFO] Matrix sparsity: 99.9998%


In [10]:
# Save processed data
processed_path = config['paths']['processed_data']

train_df.to_parquet(processed_path + 'train.parquet', index=False)
test_df.to_parquet(processed_path + 'val.parquet', index=False)
test_df.to_parquet(processed_path + 'test.parquet', index=False)

print(f"[INFO] Train saved: {train_df.shape}")
print(f"[INFO] Val saved:  {test_df.shape}")
print(f"[INFO] Test saved:  {test_df.shape}")

[INFO] Train saved: (608593, 14)
[INFO] Val saved:  (98155, 14)
[INFO] Test saved:  (98155, 14)


In [11]:
import joblib

joblib.dump(encoders, processed_path + 'encoders.pkl')
joblib.dump(scaler, processed_path + 'scaler.pkl')

print("[INFO] Encoders and scaler saved ✅")

[INFO] Encoders and scaler saved ✅


In [12]:
import joblib

joblib.dump(train_matrix, processed_path + 'user_item_train_matrix.pkl')
joblib.dump(val_matrix, processed_path + 'user_item_val_matrix.pkl')
joblib.dump(test_matrix, processed_path + 'user_item_test_matrix.pkl')


['data/processed/user_item_test_matrix.pkl']